# prepare_dataset

Ten notebook służy do przygotowania finalnego zbioru do modelowania:
- wybór jednego targetu / białka,
- zbudowanie `model_df`,
- opcjonalne ograniczenie zbioru do ok. 10k rekordów,
- random split,
- scaffold split,
- mały subset do sanity check / overfitu.

Zakładam, że w katalogu projektu masz już:
- `data_preparation.py`
- `splits.py`

i że potrafisz wczytać surowe tabele ChEMBL do zmiennych:
- `activities`
- `assays`
- `targets`
- `structures`


## 1. Importy i ustawienia

In [4]:
from pathlib import Path
import chembl_downloader
sqlite_path = chembl_downloader.download_extract_sqlite()
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("chembl-eda")
    .master("local[*]")
    .config("spark.jars.packages", "org.xerial:sqlite-jdbc:3.51.1.0")
    .getOrCreate()
)

import pandas as pd

from data_preparation import (
    cast_chembl_tables,
    build_base_table,
    filter_activity_rows,
    summarize_target_candidates,
    build_single_target_regression_dataset,
    quick_modeling_report,
)

from splits import (
    random_split,
    scaffold_split,
    split_summary,
    scaffold_overlap_report,
)

DATA_DIR = Path("prepared_data")
DATA_DIR.mkdir(exist_ok=True)

SEED = 42
STANDARD_TYPE = "IC50"
RELATION = "="
MIN_CONFIDENCE_SCORE = 8.0
ORGANISM = "Homo sapiens"

# Możesz ograniczyć finalny dataset do około 10k rekordów:
MAX_ROWS = 10_000

# Rozmiar malutkiego subsetu do sanity check / overfitu
TINY_SUBSET_SIZE = 128


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1875)
	at org.apache.spark.SparkContext.$anonfun$new$17(SparkContext.scala:551)
	at org.apache.spark.SparkContext.$anonfun$new$17$adapted(SparkContext.scala:551)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:551)
	at org.apache.spar

## 2. Wczytaj surowe tabele

Tu wklej swój blok ładowania danych z EDA albo podmień na odczyt z plików.
Po tej komórce mają istnieć zmienne:
- `activities`
- `assays`
- `targets`
- `structures`


In [2]:
# kod ładowania tabel z EDA.
# Przykład:

#
# Po uruchomieniu tej komórki powinny istnieć 4 DataFrame'y:
# activities, assays, targets, structures
# w nowszych wersjach chembl nazwy kolumn moga sie roznic

jdbc_url = f"jdbc:sqlite:{sqlite_path}"
props = {"driver": "org.sqlite.JDBC"}

def t(name, cols=None, custom_schema=None):
    reader = (spark.read.format("jdbc")
              .option("url", jdbc_url)
              .option("dbtable", name)
              .option("driver", "org.sqlite.JDBC"))
    if custom_schema:
        reader = reader.option("customSchema", custom_schema)

    df = reader.load()
    return df.select(*cols) if cols else df


activities = t(
    "activities",
    cols=[
        "activity_id","assay_id","molregno",
        "standard_type","standard_relation","standard_units",
        "standard_value","pchembl_value"
    ],
    custom_schema="""
        activity_id BIGINT,
        assay_id BIGINT,
        molregno BIGINT,
        standard_value DOUBLE,
        pchembl_value DOUBLE
    """
)

assays0 = t("assays")
assays = (assays0
          .withColumnRenamed("tid", "target_id")
          .select("assay_id", "target_id", "assay_type", "confidence_score"))

targets0 = t("target_dictionary")

targets = (targets0
           .withColumnRenamed("tid", "target_id")
           .withColumnRenamed("chembl_id", "target_chembl_id")
           .select("target_id", "target_chembl_id", "pref_name", "target_type", "organism"))

structures = t("compound_structures", ["molregno","canonical_smiles","standard_inchi_key"])
raise NotImplementedError("Wklej tutaj swój kod ładowania danych z EDA.")


NameError: name 'sqlite_path' is not defined

## 3. Cast i tabela bazowa

In [ ]:
activities_c, assays_c, targets_c, structures_c = cast_chembl_tables(
    activities, assays, targets, structures
)

base = build_base_table(activities_c, assays_c, targets_c, structures_c)
base.printSchema()


## 4. Wybór kandydatów na target

Najpierw filtrujemy sensowne rekordy, a potem patrzymy, które targety mają najwięcej danych.


In [ ]:
filtered = filter_activity_rows(
    base,
    standard_type=STANDARD_TYPE,
    relation=RELATION,
    min_confidence_score=MIN_CONFIDENCE_SCORE,
    organism=ORGANISM,
)

candidate_targets = summarize_target_candidates(filtered, top_n=30)
candidate_targets.show(30, truncate=False)


## 5. Wybierz target

Po obejrzeniu tabeli wyżej podmień `TARGET_ID`.
Dobrze, jeśli po końcowym filtrowaniu zostanie przynajmniej kilka tysięcy rekordów.


In [ ]:
TARGET_ID = None  # <- wpisz tutaj wybrany target_id

if TARGET_ID is None:
    raise ValueError("Ustaw TARGET_ID po obejrzeniu candidate_targets.show(...).")


## 6. Zbuduj finalny dataset modelowy

In [ ]:
model_df_spark = build_single_target_regression_dataset(
    activities_c,
    assays_c,
    targets_c,
    structures_c,
    target_id=TARGET_ID,
    standard_type=STANDARD_TYPE,
    relation=RELATION,
    min_confidence_score=MIN_CONFIDENCE_SCORE,
    organism=ORGANISM,
)

report = quick_modeling_report(model_df_spark)
report


In [ ]:
model_df = model_df_spark.toPandas()
print(model_df.shape)
model_df.head()


## 7. Opcjonalne ograniczenie do ~10k rekordów

Jeśli zbiór jest większy, możesz go na razie przyciąć.
Na start najprościej zrobić losową próbkę z ustalonym ziarnem.


In [ ]:
if MAX_ROWS is not None and len(model_df) > MAX_ROWS:
    model_df = model_df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

print("Final dataset shape after optional cap:", model_df.shape)
model_df[["canonical_smiles", "y"]].head()


## 8. Zapis pełnego datasetu

In [ ]:
full_dataset_path = DATA_DIR / "chembl_ic50_model_dataset.csv"
model_df.to_csv(full_dataset_path, index=False)
print(full_dataset_path)


## 9. Random split

In [ ]:
random_res = random_split(
    model_df,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    seed=SEED,
)

print(split_summary(random_res))


## 10. Scaffold split

In [ ]:
scaffold_res = scaffold_split(
    model_df,
    smiles_col="canonical_smiles",
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    seed=SEED,
)

print(split_summary(scaffold_res))
print(scaffold_overlap_report(scaffold_res))


## 11. Zapis splitów do plików

In [ ]:
random_res.train_df.to_csv(DATA_DIR / "train_random.csv", index=False)
random_res.val_df.to_csv(DATA_DIR / "val_random.csv", index=False)
random_res.test_df.to_csv(DATA_DIR / "test_random.csv", index=False)

scaffold_res.train_df.to_csv(DATA_DIR / "train_scaffold.csv", index=False)
scaffold_res.val_df.to_csv(DATA_DIR / "val_scaffold.csv", index=False)
scaffold_res.test_df.to_csv(DATA_DIR / "test_scaffold.csv", index=False)

print("Saved all split files to:", DATA_DIR.resolve())


## 12. Tiny subset do sanity check / overfitu

To będzie bardzo mały train set do pierwszego eksperymentu z MLP.
Na nim chcesz sprawdzić, czy model umie zbić train loss bardzo nisko.


In [ ]:
tiny_train_df = random_res.train_df.sample(
    n=min(TINY_SUBSET_SIZE, len(random_res.train_df)),
    random_state=SEED,
).reset_index(drop=True)

tiny_train_path = DATA_DIR / "train_random_tiny.csv"
tiny_train_df.to_csv(tiny_train_path, index=False)

print("Tiny subset shape:", tiny_train_df.shape)
print("Saved tiny subset to:", tiny_train_path)
tiny_train_df.head()


## 13. Szybka kontrola

Tu tylko upewniamy się, że wszystko wygląda sensownie:
- nie ma pustych SMILES,
- target `y` istnieje,
- liczności splitów są sensowne.


In [ ]:
print("Null SMILES in full dataset:", model_df["canonical_smiles"].isna().sum())
print("Null y in full dataset:", model_df["y"].isna().sum())

print("\nRandom split sizes:")
print(len(random_res.train_df), len(random_res.val_df), len(random_res.test_df))

print("\nScaffold split sizes:")
print(len(scaffold_res.train_df), len(scaffold_res.val_df), len(scaffold_res.test_df))


## Co dalej?

Po uruchomieniu tego notebooka będziesz mieć gotowe pliki:
- `prepared_data/chembl_ic50_model_dataset.csv`
- `prepared_data/train_random.csv`
- `prepared_data/val_random.csv`
- `prepared_data/test_random.csv`
- `prepared_data/train_scaffold.csv`
- `prepared_data/val_scaffold.csv`
- `prepared_data/test_scaffold.csv`
- `prepared_data/train_random_tiny.csv`

Następny krok:
- w `mlp_model.py` zdefiniować MLP i funkcję `smiles -> Morgan fingerprint`,
- w `train_mlp.ipynb` zacząć od `train_random_tiny.csv` i sprawdzić, czy model potrafi overfitować.
